## WavLM analysis
Loads natural and perturbed speech stimuli, extracts activations across CNN and Transformer layers of pretrained and untrained WavLM, and computes direct and indirect trajectory-curvature estimates across layers and temporal resolutions. This notebook supports the main WavLM results, stimulus-control analyses, direct-versus-indirect comparison, and temporal downsampling analyses shown in Fig. 5, Fig. 6a–b, and Etended Data Fig. 4a,c.

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd().parents[1]
audio_path = repo_root / "data" / "audios"

import torch
from transformers import WavLMModel, Wav2Vec2FeatureExtractor, AutoConfig
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = "retina"
import numpy as np
import scipy
from scipy import signal
from scipy.signal import resample_poly
from scipy.io import wavfile
import math
import statsmodels.api as sm
import os, random, pathlib, re, shutil
from fractions import Fraction

/Users/jiamingxu/miniconda3/envs/wavlm_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# helper functions
def avg_pop_curvature(X, time_axis=1, eps=1e-12):
    """
    Compute the mean turning angle of a population trajectory.

    Parameters
    ----------
    X : np.ndarray
        Population activity.
    time_axis : int
        Axis corresponding to time.
        Use 1 for arrays shaped (features, time).
        Use 0 for arrays shaped (time, features).
    eps : float
        Threshold for excluding zero-length steps.

    Returns
    -------
    float
        Mean trajectory curvature in degrees.
    """

    # Standardize to shape (features, time)
    if time_axis == 0:
        X = X.T

    V = np.diff(X, axis=1)
    norms = np.linalg.norm(V, axis=0)

    dots = np.sum(V[:, 1:] * V[:, :-1], axis=0)
    denom = norms[1:] * norms[:-1]

    valid = denom > eps

    if not np.any(valid):
        return np.nan

    cos_angles = dots[valid] / denom[valid]
    cos_angles = np.clip(cos_angles, -1.0, 1.0)

    angles = np.degrees(np.arccos(cos_angles))

    return float(np.mean(angles))

def compute_layer_curvature(layer_outputs, is_cnn=False):
    """
    Compute mean trajectory curvature for each layer (direct approach).

    Parameters
    ----------
    layer_outputs : list
        CNN or Transformer hidden-state outputs.
    is_cnn : bool
        CNN outputs are stored as tuples, so the activation tensor
        is taken from element 1.

    Returns
    -------
    np.ndarray
        Mean per-step curvature for each layer.
    """
    curvatures = []

    for layer_output in layer_outputs:
        if is_cnn:
            X = layer_output[1].squeeze(0).numpy()
        else:
            X = layer_output.squeeze(0).numpy()

        curvature = avg_pop_curvature(X, time_axis=0)
        curvatures.append(curvature)

    return np.asarray(curvatures)

def compute_layerwise_ar1(layer_outputs, is_cnn=False):
    """
    Fit AR(1) models to each unit and return the mean AR(1)
    coefficient for each layer (indirect approach)
    """
    layer_ar = []

    for layer_output in layer_outputs:
        if is_cnn:
            X = layer_output[1].squeeze(0).detach().cpu().numpy()
        else:
            X = layer_output.squeeze(0).detach().cpu().numpy()

        per_unit_ar = []

        for u in range(X.shape[1]):
            x = X[:, u] - X[:, u].mean()

            model = sm.tsa.AutoReg(
                x,
                lags=1
            ).fit()

            per_unit_ar.append(model.params[1])

        layer_ar.append(np.mean(per_unit_ar))

    return np.asarray(layer_ar)

def lowpass_and_resample(
    X,
    fs_in,
    fs_out,
    low_pass=True,
    cutoff=None,
    order=4,
    max_den=100000,
):
    """
    Optionally low-pass filter a time series, then resample it.

    Parameters
    ----------
    X : np.ndarray
        Array of shape (time, features).
    fs_in : float
        Original sampling rate in Hz.
    fs_out : float
        Target sampling rate in Hz.
    low_pass : bool
        If True, apply a low-pass filter before resampling.
    cutoff : float or None
        Low-pass cutoff frequency in Hz. If None, use 80% of
        the target Nyquist frequency.
    order : int
        Butterworth filter order.
    max_den : int
        Maximum denominator used when approximating the
        resampling ratio.

    Returns
    -------
    np.ndarray
        Resampled array of shape (time, features).
    """
    X = np.asarray(X, dtype=np.float64)

    # Optional low-pass filtering
    if low_pass:
        if cutoff is None:
            cutoff = 0.8 * (fs_out / 2.0)

        sos = butter(
            order,
            cutoff,
            btype="low",
            fs=fs_in,
            output="sos",
        )

        X = sosfiltfilt(sos, X, axis=0)

    # No resampling needed
    if np.isclose(fs_in, fs_out):
        return X

    # Compute rational resampling ratio
    ratio = Fraction(fs_out / fs_in).limit_denominator(max_den)
    up = ratio.numerator
    down = ratio.denominator

    if up < 1 or down < 1:
        raise ValueError(
            f"Invalid resampling ratio: fs_out/fs_in={fs_out / fs_in}, "
            f"approximated as {up}/{down}."
        )

    # Resample along the time axis
    Y = resample_poly(
        X,
        up=up,
        down=down,
        axis=0,
    )

    return Y

#### Load audio file 

In [ ]:
def load_mono_resampled(path, target_sr=16000):
    sr, wav = wavfile.read(path)
    dtype = wav.dtype
    wav = wav.astype(np.float32)

    # Convert stereo to mono
    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    # Scale integer audio to approximately [-1, 1]
    if np.issubdtype(dtype, np.integer):
        info = np.iinfo(dtype)
        wav = wav / max(abs(info.min), info.max)

    if sr != target_sr:
        wav = signal.resample_poly(wav, target_sr, sr)

    return torch.tensor(wav, dtype=torch.float32)

def phase_rand(waveform, seed=0):
    torch.manual_seed(seed)
    x = waveform.clone()
    X = torch.fft.rfft(x)
    mag = torch.abs(X)
    rand_phase = 2*np.pi*torch.rand_like(mag)
    rand_phase[0] = 0 # keep DC fixed
    if len(waveform)%2 == 0:
        rand_phase[-1] = 0

    X_rand = mag * torch.exp(1j * rand_phase)
    x_rand = torch.fft.irfft(X_rand, n=len(waveform))

    # match loudness
    orig_rms = torch.sqrt(torch.mean(x ** 2))
    rand_rms = torch.sqrt(torch.mean(x_rand ** 2))
    x_rand = x_rand / rand_rms * orig_rms

    return x_rand

# load original natural speech audio
waveform = load_mono_resampled(
    audio_path / "wheretheressmoke.wav",
    target_sr=16000
)
max_duration_sec = 60  # this is the length of the audio input 
waveform = waveform[: int(max_duration_sec * 16000)]
ori_rms = torch.sqrt(torch.mean(waveform ** 2))

# white noise
waveform_white = load_mono_resampled(
    audio_path / "whitenoise.wav",
    target_sr=16000
)
waveform_white = waveform_white[: int(max_duration_sec * 16000)]
white_rms = torch.sqrt(torch.mean(waveform_white ** 2))
waveform_white = waveform_white / white_rms * ori_rms

# phase randomize condition
waveform_phasernd = phase_rand(waveform)

# waveform forward condition
waveform_fwd = waveform

# waveform reverse condition
waveform_rev = waveform.flip(dims=(0,))

NameError: name 'torchaudio' is not defined

#### Load the model and feature extractor

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

model_name = "microsoft/wavlm-large"

# Load feature extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

# Pretrained WavLM-large
pre = WavLMModel.from_pretrained(
    model_name,
    output_hidden_states=True
).to(device).eval()

# Randomly initialized WavLM-large with the same architecture
cfg = AutoConfig.from_pretrained(model_name)
rnd = WavLMModel(cfg).to(device).eval()



In [ ]:
inputs_ori = feature_extractor(waveform, sampling_rate=16000, return_tensors="pt")
inputs_rev = feature_extractor(waveform_rev, sampling_rate=16000, return_tensors="pt")
inputs_white = feature_extractor(waveform_white, sampling_rate=16000, return_tensors="pt")
inputs_phasernd = feature_extractor(waveform_phasernd, sampling_rate=16000, return_tensors="pt")

inputs_ori = {k: v.to(device) for k, v in inputs_ori.items()}
inputs_rev = {k: v.to(device) for k, v in inputs_rev.items()}
inputs_white = {k: v.to(device) for k, v in inputs_white.items()}
inputs_phasernd = {k: v.to(device) for k, v in inputs_phasernd.items()}

def collect_wavlm_activations(model, inputs, hook_point="layer"):
    cnn, handles = [], []
    def hook(i):
        def fn(m, inp, out):
            x = out.detach().transpose(1,2).squeeze(0).cpu()  # [T, C]
            cnn.append((i, x))
        return fn
    for i, blk in enumerate(model.feature_extractor.conv_layers):
        target = blk if hook_point=="layer" else blk.conv
        handles.append(target.register_forward_hook(hook(i)))
    model.eval()
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, return_dict=True)
    for h in handles: h.remove()
    cnn.sort(key=lambda t: t[0])
    tr = [h.squeeze(0).cpu() for h in out.hidden_states]  # [T_enc, D]
    return cnn, tr

cnn_ori, tr_ori = collect_wavlm_activations(pre, inputs_ori, hook_point="layer")
cnn_rev, tr_rev = collect_wavlm_activations(pre, inputs_rev, hook_point="layer")
cnn_white, tr_white = collect_wavlm_activations(pre, inputs_white, hook_point="layer")
cnn_phasernd, tr_phasernd = collect_wavlm_activations(pre, inputs_phasernd, hook_point="layer")

# randomly initialized weights model
cnn_rnd, tr_rnd = collect_wavlm_activations(rnd, inputs_ori, hook_point="layer")

#### Direct curvature estimation - original sampling (Fig 5 b-c)
Compute trajectory curvature directly from consecutive activation vectors for each CNN and Transformer layer, using each layer’s original sampling rate.

In [ ]:
# Compute direct curvature for all WavLM control conditions

cnn_conditions = {
    "natural speech": cnn_ori,
    "reversed speech": cnn_rev,
    "white noise": cnn_white,
    "phase scrambled": cnn_phasernd,
    "randomly initialized model": cnn_rnd,
}

transformer_conditions = {
    "natural speech": tr_ori,
    "reversed speech": tr_rev,
    "white noise": tr_white,
    "phase scrambled": tr_phasernd,
    "randomly initialized model": tr_rnd,
}

# Compute curvature for every condition
cnn_curvature = {
    name: compute_layer_curvature(outputs, is_cnn=True)
    for name, outputs in cnn_conditions.items()
}

transformer_curvature = {
    name: compute_layer_curvature(outputs, is_cnn=False)
    for name, outputs in transformer_conditions.items()
}

In [ ]:
# Plot direct curvature across WavLM layers for all control conditions
colors = {
    "natural speech": "yellow",
    "reversed speech": "red",
    "white noise": "black",
    "phase scrambled": "green",
    "randomly initialized model": "orange",
}

n_cnn = len(cnn_curvature["natural speech"])
n_transformer = len(transformer_curvature["natural speech"])

cnn_layers = np.arange(n_cnn)
transformer_layers = np.arange(n_cnn, n_cnn + n_transformer)

plt.figure(figsize=(8, 5))

for name in cnn_conditions:
    plt.plot(
        cnn_layers,
        cnn_curvature[name],
        marker=None,
        # markersize=8,
        linewidth=2,
        color=colors[name],
    )

    plt.plot(
        transformer_layers,
        transformer_curvature[name],
        marker=None,
        # markersize=8,
        linewidth=2,
        color=colors[name],
        label=name,
    )

plt.title("Average curvature across WavLM layers")
plt.ylim(75, 135)
plt.xlabel("Layer")
plt.ylabel("Average curvature (degrees)")
plt.legend()
plt.tight_layout()

# save_path = os.path.join(fig_path, "wavLM_dir_curv_controls.pdf")
# plt.savefig(save_path, format="pdf", bbox_inches="tight")
plt.show()

#### Indirect curvature estimation - original sampling (Fig 6a)
Fit an AR(1) model to each unit’s activation time series, average the AR(1) coefficients within each layer, and convert the layer-average coefficient to implied curvature using the analytic AR(1)–curvature relationship.

In [ ]:
cnn_ar = compute_layerwise_ar1(cnn_ori, is_cnn=True)
tr_ar = compute_layerwise_ar1(tr_ori, is_cnn=False)

cnn_ind_curv = np.rad2deg(np.arccos((cnn_ar - 1) / 2))
tr_ind_curv = np.rad2deg(np.arccos((tr_ar - 1) / 2))

In [ ]:
n_cnn = len(cnn_ind_curv)
n_tr = len(tr_ind_curv)

cnn_layers = np.arange(n_cnn)
tr_layers = np.arange(n_cnn, n_cnn + n_tr)

plt.figure(figsize=(5, 5))

# Direct curvature: natural speech
plt.plot(
    cnn_layers,
    cnn_curvature["natural speech"],
    marker=None,
    linewidth=2,
    color="orange",
    label="Direct - CNN"
)

plt.plot(
    tr_layers,
    transformer_curvature["natural speech"],
    marker=None,
    linewidth=2,
    color="orange",
    label="Direct - Transformer"
)

# Indirect curvature from AR(1)
plt.plot(
    cnn_layers,
    cnn_ind_curv,
    marker=None,
    linewidth=2,
    color="black",
    label="Indirect - CNN"
)

plt.plot(
    tr_layers,
    tr_ind_curv,
    marker=None,
    linewidth=2,
    color="black",
    label="Indirect - Transformer"
)

plt.xlabel("Layer")
plt.ylabel("Curvature (degrees)")
plt.title("Direct vs. indirect curvature for natural speech")
plt.legend()
plt.tight_layout()

plt.show()

#### Direct curvature across sampling rates (Fig 6b; Extended Data Fig 4c)
Compute direct trajectory curvature across WavLM layers after resampling activations to multiple temporal resolutions. The main analysis uses resampling without low-pass filtering; the Extended Data control repeats the analysis with low-pass filtering before resampling.

In [ ]:
# Fig. 6b: low_pass = False
# Extended Data Fig. 4c: low_pass = True
low_pass = False

fs_out_list = [40, 20, 5, 0.5]

# Native sampling rates of CNN layers
fs_layers = {
    0: 3200,
    1: 1600,
    2: 800,
    3: 400,
    4: 200,
    5: 100,
    6: 50,
}

all_curvatures = {}

for fs_out in fs_out_list:

    # Downsample CNN layers
    ds_cnn_outputs = []

    for layer_idx, X in cnn_ori:
        X_np = X.detach().cpu().numpy()

        X_ds = lowpass_and_resample(
            X_np,
            fs_in=fs_layers[layer_idx],
            fs_out=fs_out,
            low_pass=low_pass,
        )

        ds_cnn_outputs.append((layer_idx, X_ds))

    # Trim CNN layers to the same length after resampling
    min_len = min(X.shape[0] for _, X in ds_cnn_outputs)
    ds_cnn_outputs = [
        (layer_idx, X[:min_len])
        for layer_idx, X in ds_cnn_outputs
    ]

    # Downsample Transformer layers
    ds_tr_outputs = []

    for layer_idx, X in enumerate(tr_ori):
        X_np = X.detach().cpu().numpy()

        X_ds = lowpass_and_resample(
            X_np,
            fs_in=50,
            fs_out=fs_out,
            low_pass=low_pass,
        )

        ds_tr_outputs.append((layer_idx, X_ds))

    # Compute direct curvature
    cnn_curv = [
        avg_pop_curvature(X, time_axis=0)
        if X.shape[0] >= 3 else np.nan
        for _, X in ds_cnn_outputs
    ]

    tr_curv = [
        avg_pop_curvature(X, time_axis=0)
        if X.shape[0] >= 3 else np.nan
        for _, X in ds_tr_outputs
    ]

    all_curvatures[fs_out] = np.asarray(cnn_curv + tr_curv)


# Plot curvature across sampling rates
n_cnn = len(cnn_curv)
n_total_layers = n_cnn + len(tr_curv)
layers = np.arange(n_total_layers)

plt.figure(figsize=(12, 8))

colors = plt.cm.viridis(
    np.linspace(0, 1, len(fs_out_list))
)

for color, fs_out in zip(colors, fs_out_list):
    plt.plot(
        layers,
        all_curvatures[fs_out],
        marker="o",
        linewidth=2,
        color=color,
        label=f"{fs_out} Hz",
    )

# Divider between CNN and Transformer layers
plt.axvline(
    n_cnn - 0.5,
    linestyle="--",
    alpha=0.5,
    color="gray",
)

analysis_label = (
    "with low-pass filtering"
    if low_pass
    else "without low-pass filtering"
)

plt.title(
    f"Direct curvature across sampling rates ({analysis_label})"
)
plt.xlabel("Layer")
plt.ylabel("Direct curvature (degrees)")
plt.xticks(layers)
plt.yticks(np.arange(75, 140, 5))
plt.legend(title="Sampling rate")
plt.tight_layout()

plt.show()